# model_experiment_DLinear

DLinear (Zeng et al. 2022, *"Are Transformers Effective for Time Series Forecasting?"*) — implemented from the paper in `src/dlinear.py` (~50 lines of PyTorch, no forecasting library).

Architecture: decompose each input window into **trend** (moving average) and **seasonal remainder**, apply one linear layer to each, sum. Channel-independent: one shared model, each of the 3331 series is forecast from its own history in a single shot — direct multi-horizon over all 39 test weeks, no recursion.

Training: per-window standardization (sales scales differ 100x across series), weighted L1 loss with `1 + 4*IsHoliday` on target weeks — the exact competition metric. Series without enough history fall back to seasonal naive.

MLflow experiment: **DLinear_Training** — runs: Preprocessing, CV_input*, Fold2, Blend_*, Final. CPU is enough (the model has ~8K parameters).

In [3]:
# Kaggle bootstrap — does nothing when running locally.
# On Kaggle: attach the competition data (Add Input) and the three
# MLFLOW_TRACKING_* secrets (Add-ons -> Secrets) before Run All.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [4]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path (local runs)

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow
from src.dlinear import build_wide, train_dlinear, forecast
from src.postprocess import naive_lag52, apply_christmas_shift, blend_holiday_naive
from src.preprocessing import BLEND_HOLIDAY_WEEKS

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
print(train.shape, test.shape)

setup_mlflow("DLinear_Training")

(421570, 5) (115064, 4)
MLflow -> https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow | experiment: DLinear_Training


## Run 1 — Preprocessing decisions

In [5]:
with mlflow.start_run(run_name="DLinear_Preprocessing"):
    mlflow.log_params({
        "series_format": "regular weekly grid per (Store, Dept); gaps inside the active range -> 0",
        "normalization": "per-window standardization, std floored at $1",
        "loss": "L1 weighted 1+4*IsHoliday on target weeks (matches WMAE)",
        "fallback": "seasonal naive (lag 52/53/51) + dept median for series without a full input window",
        "horizon": 39,
    })

🏃 View run DLinear_Preprocessing at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/27142f9af733416191adb82b0d76052e
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2


## Run 2 — Input-window ablation on Fold 1

Note the data constraint: Fold-1 train has only 91 weeks, so input 52 + horizon 39 leaves exactly ONE training window per series; input 26 gives 27 windows per series. A trade-off between window length and training data — worth a paragraph in the README.

In [6]:
def eval_fold(fold, input_size, epochs=30):
    tr, va = split_fold(train, fold)
    wide_tr = build_wide(tr)
    val_dates = sorted(va["Date"].unique())
    model = train_dlinear(wide_tr, input_size, len(val_dates), epochs=epochs, verbose=False)
    fc = forecast(model, wide_tr, input_size, len(val_dates), val_dates)
    m = va.merge(fc, on=["Store", "Dept", "Date"], how="left")
    coverage = m["pred"].notna().mean()
    m["pred"] = m["pred"].fillna(pd.Series(naive_lag52(tr, va), index=m.index))
    dep_med = tr.groupby("Dept")["Weekly_Sales"].median()
    m["pred"] = m["pred"].fillna(m["Dept"].map(dep_med)).fillna(0)
    rep = wmae_report(m["Weekly_Sales"], m["pred"], m["IsHoliday"])
    return rep, coverage, m, tr, model

results = {}
for L in (26, 39, 52):
    rep, cov, _, _, _ = eval_fold(1, L)
    results[L] = rep["wmae"]
    with mlflow.start_run(run_name=f"DLinear_CV_input{L}"):
        mlflow.log_params({"input_size": L, "fold": 1, "epochs": 30})
        mlflow.log_metrics({**rep, "coverage": cov})
    print(f"input={L}: coverage {cov:.3f}, {rep}")

BEST_INPUT = min(results, key=results.get)
print("best input size:", BEST_INPUT)

🏃 View run DLinear_CV_input26 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/9feb1d9ab11c410fa20af24c7e7a315d
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2
input=26: coverage 0.977, {'wmae': 3377.5712923196083, 'mae_holiday': 5199.055925003862, 'mae_nonholiday': 2608.1832490561637}
🏃 View run DLinear_CV_input39 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/34f842579620477e97e8d0ac6ec71d1f
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2
input=39: coverage 0.975, {'wmae': 3134.6145967068023, 'mae_holiday': 4710.97379236099, 'mae_nonholiday': 2468.766587197436}
🏃 View run DLinear_CV_input52 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/7e2a695d1b884dd896dace1cf99e1734
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-for

## Run 3 — Fold 2 sanity check

In [7]:
rep2, cov2, _, _, _ = eval_fold(2, BEST_INPUT)
with mlflow.start_run(run_name="DLinear_Fold2"):
    mlflow.log_params({"input_size": BEST_INPUT, "fold": 2})
    mlflow.log_metrics({**rep2, "coverage": cov2})
print(rep2)

🏃 View run DLinear_Fold2 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/b46a568ebb8140c6b252b3797121445e
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2
{'wmae': 1803.217221235585, 'mae_holiday': 1943.4154190685501, 'mae_nonholiday': 1765.0601780381458}


## Run 4 — Holiday blend (no Christmas — see LightGBM notebook for why)

In [8]:
rep1, _, m1, tr1, _ = eval_fold(1, BEST_INPUT)
blend_scores = {}
for w in (0.0, 0.5, 0.75):
    blended = blend_holiday_naive(m1[["Store", "Dept", "Date", "pred"]], tr1,
                                  weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    rep = wmae_report(m1["Weekly_Sales"], blended["pred"], m1["IsHoliday"])
    blend_scores[w] = rep["wmae"]
    with mlflow.start_run(run_name=f"DLinear_Blend_noXmas_w{w}"):
        mlflow.log_params({"input_size": BEST_INPUT, "blend_weight": w, "fold": 1})
        mlflow.log_metrics(rep)
    print(f"w={w}: {rep}")
BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

🏃 View run DLinear_Blend_noXmas_w0.0 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/fe240e85364b4f0c9a7c7d7a01141be5
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2
w=0.0: {'wmae': 3106.531774468392, 'mae_holiday': 4085.787118859334, 'mae_nonholiday': 2692.898115552523}
🏃 View run DLinear_Blend_noXmas_w0.5 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/35e7006d04e8479bb693c64ace1997fe
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2
w=0.5: {'wmae': 2766.835384916939, 'mae_holiday': 2941.877876295771, 'mae_nonholiday': 2692.898115552523}
🏃 View run DLinear_Blend_noXmas_w0.75 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/1db70f298bdb4dbd9c207b5c948675be
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/

## Run 5 — Final: train on ALL data, predict test, submission

In [9]:
wide_full = build_wide(train)
test_dates = sorted(test["Date"].unique())
final_model = train_dlinear(wide_full, BEST_INPUT, len(test_dates), epochs=30)

fc = forecast(final_model, wide_full, BEST_INPUT, len(test_dates), test_dates)
sub = test[["Store", "Dept", "Date"]].merge(fc, on=["Store", "Dept", "Date"], how="left")
covered = sub["pred"].notna().mean()
sub["pred"] = sub["pred"].fillna(pd.Series(naive_lag52(train, test), index=sub.index))
dep_med = train.groupby("Dept")["Weekly_Sales"].median()
sub["pred"] = sub["pred"].fillna(sub["Dept"].map(dep_med)).fillna(0)

sub = apply_christmas_shift(sub, pred_col="pred")
if BLEND_W > 0:
    sub = blend_holiday_naive(sub, train, weight=BLEND_W, holiday_dates=BLEND_HOLIDAY_WEEKS)

with mlflow.start_run(run_name="DLinear_Final"):
    mlflow.log_params({"input_size": BEST_INPUT, "epochs": 30, "blend_weight": BLEND_W,
                       "post": "christmas_shift + noXmas_blend"})
    mlflow.log_metrics({"fold1_wmae": results[BEST_INPUT], "test_coverage": covered})
    mlflow.pytorch.log_model(final_model, name="model",
                             registered_model_name="walmart-dlinear")

make_submission(sub, "pred", "submission_dlinear.csv")
print(f"wrote submission_dlinear.csv (coverage {covered:.3f}, blend w={BLEND_W})")

  epoch 10/30  weighted-L1 2257.0
  epoch 20/30  weighted-L1 2256.2
  epoch 30/30  weighted-L1 2255.9
🏃 View run DLinear_Final at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/bf415a206d0e492c819d11f52031b219
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2


MlflowException: If `serialization_format` is set to 'pt2', then input_example is required. It must be a numpy array or torch tensor, or a tuple/list of numpy arrays or torch tensors. This is because 'pt2' is a traced-graph format: PyTorch traces the model graph by virtually executing model.forward with the provided example input.

In [10]:
import numpy as np

with mlflow.start_run(run_name="DLinear_Final"):
    mlflow.log_params({"input_size": BEST_INPUT, "epochs": 30, "blend_weight": BLEND_W,
                       "post": "christmas_shift + noXmas_blend"})
    mlflow.log_metrics({"fold1_wmae": results[BEST_INPUT], "test_coverage": covered})
    mlflow.pytorch.log_model(final_model, name="model",
                             input_example=np.zeros((2, BEST_INPUT), dtype=np.float32),
                             registered_model_name="walmart-dlinear")

make_submission(sub, "pred", "submission_dlinear.csv")
print(f"wrote submission_dlinear.csv (blend w={BLEND_W})")

2026/07/10 13:10:24 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/10 13:10:37 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.25.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torchvision==0.25.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
/usr/local/lib/python3.12/dist-packages/torch/export/pt2_archive/_package.py:785: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-w

🏃 View run DLinear_Final at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2/runs/ddbe2bae9be14527b222bcc94b0b75f4
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/2
wrote submission_dlinear.csv (blend w=0.75)
